# Entrenar YOLO26 para UNEMI Guide AI (Roboflow + Colab)

Este cuaderno entrena el modelo que reconoce **en qué zona del campus** está el estudiante a partir de una foto.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

Ejecuta las celdas **en orden** (botón ▶ de cada una). Al final se descarga `modelo_unemi.zip` con `best.pt` y `data.yaml`: eso es lo que se instala en la app.

Guía completa: `docs/entrenar_yolo_roboflow.md` del proyecto.

## 1. Comprobar la GPU e instalar las librerías

In [ ]:
!nvidia-smi -L || echo "⚠️ SIN GPU: ve a Entorno de ejecución → Cambiar tipo → GPU (T4)"
!pip install -q -U ultralytics roboflow

## 2. Descargar tus fotos etiquetadas desde Roboflow

En Roboflow: tu proyecto → **Versions** → la versión que generaste → **Download Dataset** → formato **YOLO26** → **Show download code** → copia el código de *Jupyter*.

Pega ese código **en la celda de abajo**, reemplazando las líneas de ejemplo. Debe terminar creando la variable `dataset`.

In [ ]:
# ⬇️ BORRA estas líneas de ejemplo y PEGA aquí el código que te da Roboflow ⬇️
from roboflow import Roboflow
rf = Roboflow(api_key="PEGA_AQUI_TU_API_KEY")
project = rf.workspace("unemi-guide-ai").project("unemi-zonas")
version = project.version(1)
dataset = version.download("yolov26")  # usa el formato que diga Roboflow

print("Fotos descargadas en:", dataset.location)

## 3. Revisar que los nombres de las clases sean zonas de la app

Roboflow quita tildes y paréntesis (`Bloque T -FACI-`); la app los reconoce igual. Si sale ❌, la clase no corresponde a ninguna zona: corrígela en Roboflow (*Classes & Tags*), genera una versión nueva y repite el paso 2.

In [ ]:
import re, unicodedata, yaml

# Clases del proyecto de Roboflow "unemi-zonas" (sin tildes ni paréntesis).
CLASES_DE_LA_APP = [
    "Bloque B",
    "Bloque C",
    "Bloque E",
    "Bloque F",
    "Bloque H",
    "Bloque I",
    "Bloque J",
    "Bloque K",
    "Bloque L",
    "Bloque M",
    "Bloque N",
    "Bloque O",
    "Bloque P",
    "Bloque R",
    "Bloque S",
    "Bloque T -FACI-",
    "Bloque U",
    "Bloques V-W-X",
    "CRAI",
    "Posgrado",
    "SIM",
    "CDIP",
    "Bloque de Investigacion de Biotecnologia",
    "Estadio Universitario UNEMI",
    "Piscina UNEMI",
    "Entrada de vehiculos",
    "Area verde -Bloques R-U-",
    "Area verde sur",
    "Estacionamiento 1",
    "Estacionamiento 2"
]

def clave(nombre):
    """Igual que la app: ignora tildes, signos, espacios y mayúsculas."""
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", sin_tildes.lower())

CLAVES = {clave(n) for n in CLASES_DE_LA_APP}

with open(f"{dataset.location}/data.yaml", encoding="utf-8") as f:
    datos = yaml.safe_load(f)
nombres = datos["names"].values() if isinstance(datos["names"], dict) else datos["names"]

todo_bien = True
for nombre in nombres:
    if clave(nombre) in CLAVES:
        print("✅", nombre)
    else:
        todo_bien = False
        print("❌", repr(nombre), "-> no es una zona de la app. Revisa cómo está escrito.")
print()
print("Todo listo para entrenar 👍" if todo_bien else "⚠️ Corrige los nombres con ❌ antes de entrenar.")

## 4. Entrenar (20–60 minutos)

`yolo26n.pt` es el modelo más pequeño: el servidor tiene poca memoria. Si el entrenamiento deja de mejorar, se detiene solo (`patience`).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
resultados = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    patience=20,
    name="unemi_zonas",
    exist_ok=True,
)

## 5. ¿Quedó bien?

- **mAP50 > 0.8** → bueno. Entre 0.6 y 0.8 → aceptable para una demo. Menos de 0.6 → faltan fotos o están mal etiquetadas.
- En la **matriz de confusión** mira qué zonas confunde entre sí: a esas les faltan fotos.

In [ ]:
from IPython.display import Image, display
carpeta = "runs/detect/unemi_zonas"
metricas = YOLO(f"{carpeta}/weights/best.pt").val(data=f"{dataset.location}/data.yaml", verbose=False)
print(f"mAP50 = {metricas.box.map50:.2f}   |   mAP50-95 = {metricas.box.map:.2f}")
display(Image(f"{carpeta}/results.png", width=900))
display(Image(f"{carpeta}/confusion_matrix.png", width=700))

## 6. Probar con las fotos de *test* (fotos que el modelo nunca vio)

In [ ]:
import glob
modelo = YOLO(f"{carpeta}/weights/best.pt")
fotos = sorted(glob.glob(f"{dataset.location}/test/images/*"))[:6]
for i, foto in enumerate(fotos):
    r = modelo(foto, conf=0.25, verbose=False)[0]
    if len(r.boxes):
        mejor = int(r.boxes.conf.argmax())
        print(f"{foto.split('/')[-1]:40s} -> {r.names[int(r.boxes.cls[mejor])]} ({float(r.boxes.conf[mejor]):.0%})")
    else:
        print(f"{foto.split('/')[-1]:40s} -> (no reconoció ninguna zona)")
    display(Image(r.save(filename=f"/content/prueba_{i}.jpg"), width=500))

## 7. Descargar el modelo

Se descarga **`modelo_unemi.zip`**. Envíalo junto con 3–5 fotos del campus que **no** hayas subido a Roboflow, para probarlo antes de instalarlo en la app.

In [ ]:
import shutil, os
os.makedirs("modelo_unemi", exist_ok=True)
shutil.copy(f"{carpeta}/weights/best.pt", "modelo_unemi/best.pt")
shutil.copy(f"{dataset.location}/data.yaml", "modelo_unemi/data.yaml")
for extra in ["results.png", "confusion_matrix.png"]:
    if os.path.exists(f"{carpeta}/{extra}"):
        shutil.copy(f"{carpeta}/{extra}", f"modelo_unemi/{extra}")
shutil.make_archive("modelo_unemi", "zip", "modelo_unemi")

from google.colab import files
files.download("modelo_unemi.zip")